In [ ]:
!pip install fiftyone

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
classes = ["rose", "orange", "car"]

In [3]:
train_path = "/content/drive/MyDrive/fiftyone/train"

In [4]:
test_path = "/content/drive/MyDrive/fiftyone/test"

The downloading of photos. If you don't put dataset.delete(), you have to reload the environment or try deleting the dataset because it caches the photos and acts like it downloaded another class

In [37]:
# Prepare to download specific class from open images
import fiftyone
import fiftyone.zoo as zoo

number_of_samples = 61

print("Downloading is starting...")

dataset = zoo.load_zoo_dataset(
    "open-images-v7",
    split="train",
    max_samples=number_of_samples,
    shuffle=True,
    classes=["Orange"],
    label_types=["segmentations"]
)

dataset.export(
    export_dir=test_path + '/' + "Orange".lower(),
    dataset_type=fiftyone.types.ImageSegmentationDirectory,
)

print("Downloading ended.")


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/v5/train-masks/train-masks-5.zip' to '/root/fiftyone/open-images-v7/train/labels/masks/5.zip'


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/v5/train-masks/train-masks-4.zip' to '/root/fiftyone/open-images-v7/train/labels/masks/4.zip'


Found 13 images, downloading the remaining 48


INFO:fiftyone.utils.openimages:Found 13 images, downloading the remaining 48


...

Dataset 'open-images-v7-train-61' created


INFO:fiftyone.zoo.datasets:Dataset 'open-images-v7-train-61' created


Directory '/content/drive/MyDrive/fiftyone/test/orange' already exists; export will be merged with existing files


 100% |███████████████████| 61/61 [1.5s elapsed, 0s remaining, 42.1 samples/s]         


INFO:eta.core.utils: 100% |███████████████████| 61/61 [1.5s elapsed, 0s remaining, 42.1 samples/s]         


In [ ]:
import fiftyone
import fiftyone.zoo as zoo
import fiftyone.types as fot
import shutil

# Number of samples per class
number_of_samples = 90


print("Downloading is starting...")

for class_name in classes:
    print(f"Downloading class: {class_name}")

    dataset = zoo.load_zoo_dataset(
        "open-images-v7",
        split="train",
        max_samples=number_of_samples,
        shuffle=True,
        classes=[class_name],
        label_types=["segmentations"]
    )

    export_dir = f"{test_path}/{class_name.lower()}"
    dataset.export(
        export_dir=export_dir,
        dataset_type=fot.ImageSegmentationDirectory,
    )

    dataset.delete()

print("Downloading ended.")

Imports

In [22]:
import os
import random
import logging
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
from pathlib import Path
from PIL import Image
from sklearn.metrics import f1_score

Dataset for the photos

In [23]:
class MultiClassFolderDataset(Dataset):
    def __init__(self, root_dir, mode='train', target_size=(256, 256), transform=None, mask_transform=None):
        self.mode = mode
        self.target_size = target_size
        self.class_names = []

        class_dirs = [d for d in os.listdir(Path(root_dir) / mode) if not d.startswith('.')]
        self.num_classes = len(class_dirs)
        self.class_dirs = sorted(class_dirs)
        self.class_names = self.class_dirs
        self.image_index = {}

        for class_idx, class_name in enumerate(self.class_dirs):
            data_dir = Path(root_dir) / mode / class_name / "data"
            label_dir = Path(root_dir) / mode / class_name / "labels"

            image_files = sorted(os.listdir(data_dir))
            mask_files = sorted(os.listdir(label_dir))

            for img_file, mask_file in zip(image_files, mask_files):
                img_path = data_dir / img_file
                mask_path = label_dir / mask_file

                base_name = img_path.stem
                mask_path = label_dir / f"{base_name}.png"

                if not mask_path.exists():
                    continue

                self.image_index[base_name] = {
                    "image_path": img_path,
                    "mask_path": mask_path,
                    "class_idx": class_idx + 1
                }

        self.samples = list(self.image_index.values())

        self.transform = transform
        self.mask_transform = mask_transform
        self.basic_transform = transforms.ToTensor()

    def __len__(self):
        return len(self.samples)

    def augment(self, image, masks):
        if random.random() > 0.5:
            image = TF.hflip(image)
            masks = [TF.hflip(mask) for mask in masks]

        if random.random() > 0.5:
            image = TF.vflip(image)
            masks = [TF.vflip(mask) for mask in masks]

        angle = random.uniform(-5, 5)
        image = TF.rotate(image, angle, interpolation=transforms.InterpolationMode.BILINEAR)

        masks = [mask.to(torch.uint8) for mask in masks]
        masks = [TF.to_pil_image(mask) for mask in masks]
        masks = [TF.rotate(mask, angle, interpolation=transforms.InterpolationMode.NEAREST) for mask in masks]

        masks = [transforms.ToTensor()(mask) for mask in masks]

        return image, masks

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image_path = sample["image_path"]
        mask_path = sample["mask_path"]
        class_idx = sample["class_idx"]

        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        image = image.resize(self.target_size, Image.BILINEAR)
        mask = mask.resize(self.target_size, Image.NEAREST)

        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        if self.mask_transform:
            mask = self.mask_transform(mask)
        else:
            mask = transforms.ToTensor()(mask)

        mask = mask.squeeze(0).long()
        mask = (mask > 0).long() * class_idx

        return image, mask


Model for segmentation

In [24]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=4):
        super().__init__()
        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool1(d1))
        d3 = self.down3(self.pool2(d2))
        d4 = self.down4(self.pool3(d3))
        bn = self.bottleneck(self.pool4(d4))

        up4 = self.up4(bn)
        up4 = torch.cat([up4, d4], dim=1)
        up4 = self.conv4(up4)

        up3 = self.up3(up4)
        up3 = torch.cat([up3, d3], dim=1)
        up3 = self.conv3(up3)

        up2 = self.up2(up3)
        up2 = torch.cat([up2, d2], dim=1)
        up2 = self.conv2(up2)

        up1 = self.up1(up2)
        up1 = torch.cat([up1, d1], dim=1)
        up1 = self.conv1(up1)

        return self.final(up1)


Lots of logic for saving and loading the model, used when needed

In [9]:
path = "/content/drive/MyDrive/SegNet2.pth"

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [41]:
root_dir = "/content/drive/MyDrive/fiftyone"

train_dataset = MultiClassFolderDataset(root_dir, mode="train", target_size = (256, 256))
val_dataset = MultiClassFolderDataset(root_dir, mode="test", target_size = (256, 256))

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [42]:
model = UNet(3, 4).to(device)
checkpoint = torch.load(path)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
path = "/content/drive/MyDrive/SegNet2.pth"

torch.save({'model_state_dict': model.state_dict()}, path)

In [ ]:
model = UNet(3, 4).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

Training and validating

In [43]:
import os
import torch
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_epochs = 20
best_val_loss = float('inf')
best_val_loss = 0.5

save_path = '/content/drive/MyDrive/SegNet2.pth'

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    model.train()
    running_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Validating Epoch {epoch+1}"):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({'model_state_dict': model.state_dict()}, save_path)
        print(f"✅ Model saved at '{save_path}' with val loss {avg_val_loss:.4f}")



Epoch 1/20


Validating Epoch 1: 100%|██████████| 18/18 [00:08<00:00,  2.07it/s]


Train Loss: 0.3091, Val Loss: 0.5576

Epoch 2/20


Validating Epoch 2: 100%|██████████| 18/18 [00:06<00:00,  2.73it/s]


Train Loss: 0.3078, Val Loss: 0.4993
✅ Model saved at '/content/drive/MyDrive/SegNet2.pth' with val loss 0.4993

Epoch 3/20


...

Train Loss: 0.1729, Val Loss: 0.5652

Epoch 19/20


Validating Epoch 19: 100%|██████████| 18/18 [00:06<00:00,  2.71it/s]


Train Loss: 0.1542, Val Loss: 0.5765

Epoch 20/20


Validating Epoch 20: 100%|██████████| 18/18 [00:08<00:00,  2.09it/s]

Train Loss: 0.1465, Val Loss: 0.5817


Computing metrics

In [30]:
import numpy as np
import torch
from sklearn.metrics import f1_score

def compute_metrics(model, loader, device, num_classes=4):
    model.eval()
    total_intersection = np.zeros(num_classes, dtype=np.float64)
    total_union = np.zeros(num_classes, dtype=np.float64)
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            preds_np, masks_np = preds.cpu().numpy(), masks.cpu().numpy()
            all_preds.append(preds_np.flatten())
            all_labels.append(masks_np.flatten())

            for cls in range(1, num_classes):
                pred_cls = (preds_np == cls).astype(np.uint8)
                mask_cls = (masks_np == cls).astype(np.uint8)

                total_intersection[cls] += np.sum(pred_cls * mask_cls)
                total_union[cls] += np.sum(pred_cls) + np.sum(mask_cls)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    dice_scores = (2 * total_intersection + 1e-5) / (total_union + total_intersection + 1e-5)
    mean_dice = dice_scores[1:].mean()

    micro_f1 = f1_score(all_labels, all_preds, average='micro', labels=list(range(1, num_classes)))
    macro_f1 = f1_score(all_labels, all_preds, average='macro', labels=list(range(1, num_classes)))

    print(f"\nMean Dice (excluding background): {mean_dice:.4f}")
    print(f"Micro F1 (excluding background): {micro_f1:.4f}")
    print(f"Macro F1 (excluding background): {macro_f1:.4f}")

    return mean_dice, micro_f1, macro_f1


In [31]:
mean_dice, micro_f1, macro_f1 = compute_metrics(model, val_loader, device)


Mean Dice (excluding background): 0.4731
Micro F1 (excluding background): 0.6222
Macro F1 (excluding background): 0.6222


API for UI

In [ ]:
!pip install flask-ngrok


In [ ]:
def colorize_mask(mask):
    colormap = {
    0: (0, 0, 0),        # Background - Black
    1: (0, 0, 255),      # car -> Blue
    2: (255, 165, 0),    # orange -> Orange
    3: (255, 0, 0),      # rose -> Red
    }

    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)

    for class_idx, color in colormap.items():
        color_mask[mask == class_idx] = color

    return color_mask

In [ ]:
from flask import Flask, request, render_template, redirect, url_for
import os
import torch
import numpy as np
from PIL import Image
from google.colab.output import eval_js

app = Flask(__name__, static_folder='/content/drive/MyDrive/static', template_folder='/content/drive/MyDrive/templates')

UPLOAD_FOLDER = '/content/drive/MyDrive/static/uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = UNet(3, 4).to(device)
checkpoint = torch.load(path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

def preprocess_image(img):
    img = img.resize((256, 256))
    img = np.asarray(img).astype(np.float32) / 255.0
    img = img.transpose(2, 0, 1) 
    img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
    return img.to(device)

def predict_mask(tensor):
    with torch.no_grad():
        output = model(tensor)
        pred = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
    return pred

@app.route('/', methods=['GET', 'POST'])
def upload_file():
    if request.method == 'POST':
        file = request.files['file']
        if file:
            img_path = os.path.join(UPLOAD_FOLDER, file.filename)
            file.save(img_path)

            image = Image.open(img_path).convert('RGB')
            tensor = preprocess_image(image)

            mask = predict_mask(tensor)

            color_mask = colorize_mask(mask)
            mask_img = Image.fromarray(color_mask)
            mask_save_path = os.path.join(UPLOAD_FOLDER, "mask_" + file.filename)
            mask_img.save(mask_save_path)

            return redirect(url_for('result', input_image=file.filename, mask_image="mask_" + file.filename))
    return render_template('index.html')

@app.route('/result')
def result():
    input_image = request.args.get('input_image')
    mask_image = request.args.get('mask_image')
    return render_template('result.html', input_image=input_image, mask_image=mask_image)

if __name__ == '__main__':
    print(eval_js("google.colab.kernel.proxyPort(5000)"))
    app.run(port=5000)
